# 🏦 Module 7 — Advanced 2026: Reasoning, MCP & Multi-Agents
**Domain:** Banking CRM · **Dataset:** CFPB Consumer Complaints · Telco Churn · Synthetic Bank Statements  
**Duration:** ~6 hours across 3 lessons

---
| Lesson | Title | Key Concepts |
|--------|-------|--------------|
| 07a | Reasoning Models & Extended Thinking | o3, Claude extended thinking, cost tradeoff |
| 07b | Multimodal Vision LLMs | GPT-4o vision, document extraction, GDPR note |
| 07c | Production Multi-Agent Systems | LangGraph orchestrator, guardrails, audit log |

---
> 💡 **Setup:** `pip install -r requirements.txt` · LM Studio running on `localhost:1234` for local inference


In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import os, json, time, base64, datetime
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY and ANTHROPIC_API_KEY from .env

# ─ LLM clients ───────────────────────────────────────────────────
from openai import OpenAI
from anthropic import Anthropic

openai_client  = OpenAI()
claude_client  = Anthropic()

# ─ LM Studio (free local inference) ──────────────────────────────
lm_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# ─ Course checker ────────────────────────────────────────────────
from llm_checker import check, hint, evaluate, progress, show_exercise

print("✅  All imports OK — Module 7 ready")

---
## Lesson 07a — Reasoning Models & Extended Thinking (~1.5 h)
### When and How to Use Reasoning Models

**Learning objectives:**
- Understand how extended chain-of-thought (CoT) works inside reasoning models
- Use Claude's `extended_thinking` / `thinking` parameter
- Build a cost × quality decision framework from real benchmarks

**Key concepts:**
- `thinking={'type': 'enabled', 'budget_tokens': N}` — Claude Extended Thinking
- Reasoning tokens are billed separately; each task has a different sweet spot
- 5-criteria decision framework: ambiguity · steps · stakes · cost tolerance · latency budget


In [ ]:
# ── Load CFPB dataset ─────────────────────────────────────────────
from datasets import load_dataset
import pandas as pd

ds = load_dataset("cfpb/us-consumer-finance-complaints", split="train").select(range(1000))
df = pd.DataFrame(ds)
print(df.columns.tolist())
print(df[["complaint_what_happened","product"]].dropna().head(3))

In [ ]:
# ── Example: Standard Claude call ────────────────────────────────
def classify_complaint_standard(complaint_text: str) -> dict:
    """Classify a CFPB complaint using standard Claude (no extended thinking)."""
    t0 = time.perf_counter()
    msg = claude_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": (
                "Classify this bank complaint into: [credit_card, mortgage, checking_account, "
                "loan, other]. Reply JSON: {category, confidence, reason}\n\n"
                f"COMPLAINT: {complaint_text[:600]}"
            )
        }]
    )
    elapsed = time.perf_counter() - t0
    try:
        data = json.loads(msg.content[0].text)
    except Exception:
        data = {"category": "parse_error", "confidence": 0, "reason": msg.content[0].text}
    data["latency_s"] = round(elapsed, 2)
    data["input_tokens"] = msg.usage.input_tokens
    data["output_tokens"] = msg.usage.output_tokens
    return data

# Quick smoke test
sample = df["complaint_what_happened"].dropna().iloc[0]
result = classify_complaint_standard(sample)
print(result)

In [ ]:
# ── Example: Claude Extended Thinking ────────────────────────────
def classify_complaint_extended(complaint_text: str, budget: int = 5000) -> dict:
    """Classify using Claude Extended Thinking — deeper reasoning, higher cost."""
    t0 = time.perf_counter()
    msg = claude_client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=8000,
        thinking={"type": "enabled", "budget_tokens": budget},
        messages=[{
            "role": "user",
            "content": (
                "Analyze this complex bank complaint. Consider regulatory implications, "
                "root cause, and classify into: [credit_card, mortgage, checking_account, loan, other]. "
                "Reply JSON: {category, confidence, regulatory_flag, root_cause, reasoning_summary}\n\n"
                f"COMPLAINT: {complaint_text[:1200]}"
            )
        }]
    )
    elapsed = time.perf_counter() - t0
    # Separate thinking block from text block
    text_block = next((b for b in msg.content if b.type == "text"), None)
    think_block = next((b for b in msg.content if b.type == "thinking"), None)
    try:
        data = json.loads(text_block.text)
    except Exception:
        data = {"category": "parse_error", "confidence": 0}
    data["latency_s"] = round(elapsed, 2)
    data["thinking_tokens"] = len(think_block.thinking.split()) * 1.3 if think_block else 0
    data["input_tokens"]  = msg.usage.input_tokens
    data["output_tokens"] = msg.usage.output_tokens
    return data

sample_complex = df["complaint_what_happened"].dropna().iloc[5]
result_ext = classify_complaint_extended(sample_complex)
print(result_ext)

---
### 🟡 EXERCISE Ex 07a-1 — Standard vs Extended Thinking Benchmark

Select **10 CFPB complaint tasks** — 5 simple (classify category) and 5 complex (analyze regulatory implication, multi-step reasoning). Run each through:
1. Claude standard (`claude-3-5-sonnet`)
2. Claude Extended Thinking (`claude-3-7-sonnet`, `budget_tokens=5000`)

Score quality with **LLM-as-judge** (1–5). Record cost and latency.

**Hints:**
- `thinking={'type': 'enabled', 'budget_tokens': 5000}`
- Use the same prompt → both models → judge both outputs
- Cost = `(input_tokens × 0.000003) + (output_tokens × 0.000015)` for claude-3-5-sonnet

**Auto-check verifies:**
- ✓ 10 tasks run on both configurations
- ✓ Table: `task | std_score | thinking_score | cost_ratio | latency_ratio`
- ✓ Conclusion: for which task types does extended thinking justify 3–5× cost?


In [ ]:
show_exercise(
    "07a-1", "Standard vs Extended Thinking Benchmark",
    "Run 5 simple + 5 complex CFPB tasks through standard and extended thinking Claude. "
    "Score with LLM-as-judge, record cost and latency, draw conclusions.",
    hints=[
        "▸ thinking={'type': 'enabled', 'budget_tokens': 5000}",
        "▸ Same input → both models → judge both outputs",
        "▸ Simple tasks: category classification · Complex tasks: regulatory implication analysis"
    ],
    checks=[
        "10 tasks run on both configurations",
        "Table: task | std_score | thinking_score | cost_ratio | latency_ratio",
        "Conclusion: for which task types does extended thinking justify 3–5× cost?"
    ],
    exercise_type="EXERCISE"
)

In [ ]:
# ── YOUR SOLUTION — Ex 07a-1 ─────────────────────────────────────

# Claude pricing (USD per 1k tokens, approximate)
PRICING = {
    "claude-3-5-sonnet-20241022": {"input": 0.003,  "output": 0.015},
    "claude-3-7-sonnet-20250219": {"input": 0.003,  "output": 0.015},   # thinking tokens billed as output
}

def token_cost(model, input_tok, output_tok):
    p = PRICING.get(model, {"input": 0.003, "output": 0.015})
    return round((input_tok * p["input"] + output_tok * p["output"]) / 1000, 6)

def llm_judge(task: str, std_response: str, ext_response: str) -> dict:
    """Score both responses 1-5 using Claude as judge."""
    prompt = (
        f"TASK: {task}\n\n"
        f"RESPONSE A (standard): {std_response}\n\n"
        f"RESPONSE B (extended thinking): {ext_response}\n\n"
        "Score each response 1-5 on accuracy + depth + actionability. "
        "Reply JSON: {{std_score: int, ext_score: int, reasoning: str}}"
    )
    msg = claude_client.messages.create(
        model="claude-3-5-sonnet-20241022", max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        return json.loads(msg.content[0].text)
    except Exception:
        return {"std_score": 3, "ext_score": 3, "reasoning": "parse error"}

# ── Define 10 tasks ───────────────────────────────────────────────
sample_rows = df["complaint_what_happened"].dropna().iloc[:50]

TASKS = []
# 5 simple: plain classification
simple_prompts = [
    ("simple", f"Classify complaint type (one word): {row[:400]}")
    for row in sample_rows[:5]
]
# 5 complex: regulatory analysis
complex_prompts = [
    ("complex", f"Analyze regulatory implications and root cause of this CFPB complaint: {row[:800]}")
    for row in sample_rows[5:10]
]
TASKS = simple_prompts + complex_prompts

# ── Run benchmark ─────────────────────────────────────────────────
results = []

# TODO: Implement the benchmark loop
# For each task:
#   1. Call classify_complaint_standard(task_text)
#   2. Call classify_complaint_extended(task_text)
#   3. Call llm_judge(task_text, std_response, ext_response)
#   4. Compute cost_ratio = ext_cost / std_cost
#   5. Compute latency_ratio = ext_latency / std_latency
#   6. Append to results

# YOUR CODE HERE
# ─────────────────────────────────────────────────────────────────

print("Results collected:", len(results))

In [ ]:
# ── Auto-check Ex 07a-1 ───────────────────────────────────────────
check([
    (len(results) == 10,                    "10 tasks benchmarked"),
    (all("std_score"  in r for r in results), "std_score in every result"),
    (all("ext_score"  in r for r in results), "ext_score in every result"),
    (all("cost_ratio" in r for r in results), "cost_ratio in every result"),
    (all("latency_ratio" in r for r in results), "latency_ratio in every result"),
], exercise_id="07a-1")

In [ ]:
# ── Print benchmark table ─────────────────────────────────────────
if results:
    print(f"{'#':<3} {'Type':<8} {'std':>5} {'ext':>5} {'cost×':>7} {'lat×':>7}")
    print("─" * 40)
    for i, r in enumerate(results):
        print(f"{i+1:<3} {r.get('task_type','?'):<8} "
              f"{r.get('std_score',0):>5.1f} {r.get('ext_score',0):>5.1f} "
              f"{r.get('cost_ratio',0):>7.2f} {r.get('latency_ratio',0):>7.2f}")
    avg_simple  = [r for r in results if r.get("task_type") == "simple"]
    avg_complex = [r for r in results if r.get("task_type") == "complex"]
    print("\n📊 Decision framework:")
    print(f"  Simple tasks  — avg ext/std quality delta: "
          f"{sum(r.get('ext_score',0)-r.get('std_score',0) for r in avg_simple)/max(1,len(avg_simple)):+.2f}")
    print(f"  Complex tasks — avg ext/std quality delta: "
          f"{sum(r.get('ext_score',0)-r.get('std_score',0) for r in avg_complex)/max(1,len(avg_complex)):+.2f}")

---
## ✅ Lesson 07a Readiness Checklist
- ☐ 10 tasks benchmarked: standard vs extended thinking
- ☐ Cost/quality table filled
- ☐ Personal decision framework written (when to use reasoning models)